# NodeGAM: Additive oblivious decision trees

NodeGAM stacks differentiable oblivious trees with sparse feature selectors. GAM mode learns univariate terms; GA2M-style settings can learn pairwise terms.


## Model


$$
\eta(x)=\beta_0+\sum_{t=1}^{T}g_t(x_{S_t}),
\qquad |S_t|\in\{1,2\}.
$$

Each tree uses the same split feature at a given depth, while sparse selector activations concentrate mass on a small feature set.


## Shared estimator API

All neural estimators use `fit`, `predict`, `score`, `evaluate`, and
`predict_components`. The component result reconstructs predictions on the link
scale and supports shared term-importance and plotting utilities. Constructor
options such as `numerical_method` and `categorical_method` are forwarded to
PreTab and are fitted on training rows only.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = False


## Construct the estimator


In [ ]:
from nampy.models import NodeGAMClassifier, NodeGAMLSS, NodeGAMRegressor


model = NodeGAMRegressor(
    num_trees=32,
    num_layers=2,
    depth=3,
    selector_activation="entmax15",
    bin_activation="entmoid15",
    interaction_degree=2,
    l2_interactions=1e-4,
)
model.get_params(deep=False)


## Fit and inspect

Enable `RUN_TRAINING` above for a short demonstration. Real work should use a
larger validation set, enough epochs, and early stopping.


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train,
        y_train,
        max_epochs=3,
        batch_size=64,
        random_state=7,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predictions = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    metrics = model.evaluate(X_test, y_test)
    components = model.predict_components(X_test, center=True)
    components.validate_additive_reconstruction()
    display({"R2": r2, **metrics})
    display(model.term_importance(X_test).head())


## Model-specific controls

NodeGAM supports optional masked-reconstruction pretraining and recent-checkpoint averaging through fit-time controls.


In [ ]:
if RUN_TRAINING:
    pretrained = NodeGAMRegressor(num_trees=32, num_layers=2, depth=3)
    pretrained.fit(
        X_train, y_train,
        pretrain_epochs=2,
        average_checkpoints=True,
        n_last_checkpoints=2,
        max_epochs=3,
        batch_size=64,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    display(pretrained.interaction_importance(X_test))


## Task variants and limits

NodeGAM has regressor, classifier, and LSS variants. Quantile preprocessing controls are ordinary PreTab parameters.
